# Quickstart: Google WeatherNext 3 forecasts, virtual - dynamical.org Icechunk Zarr

WeatherNext 3 provides precomputed ensemble statistics for two forecast horizons and two output heads, with a 2026 archive and ongoing updates. The 15-day products initialize every six hours with hourly leads through +360 h; the 48-hour products initialize hourly with leads through +48 h.

The 0.1° head contains gridded surface weather fields. The 0.05° head provides station-trained temperature and dewpoint, evaluated globally: it is neither a resampling of the 0.1° fields nor station observations.

Dataset documentation:
- Google WeatherNext 3 forecast, 15 day, 0.1 degree, virtual: https://dynamical.org/catalog/google-weathernext3-forecast-15-day-0-1-degree-virtual/
- Google WeatherNext 3 forecast, 15 day, 0.05 degree, virtual: https://dynamical.org/catalog/google-weathernext3-forecast-15-day-0-05-degree-virtual/
- Google WeatherNext 3 forecast, 48 hour, 0.1 degree, virtual: https://dynamical.org/catalog/google-weathernext3-forecast-48-hour-0-1-degree-virtual/
- Google WeatherNext 3 forecast, 48 hour, 0.05 degree, virtual: https://dynamical.org/catalog/google-weathernext3-forecast-48-hour-0-05-degree-virtual/

Real-time data and forecasts are subject to Google's [GDM Real-Time Weather Forecasting Experimental Data Terms of Use](https://storage.googleapis.com/weathernext-public/terms-of-use.pdf); historical data are licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Retain the dataset attribution and [third-party acknowledgements](https://storage.googleapis.com/weathernext-public/weathernext-3-attributions-acknowledgements.pdf).

In [ ]:
# If running locally, follow README.md for dependency installation.
# If using Google Colab, run this cell.
!uv pip install 'dynamical-catalog>=1.0.0' 'icechunk>=2.1.2' matplotlib cartopy pandas xarray 'zarr>=3.0'

In [ ]:
import cartopy.crs as ccrs
import dynamical_catalog
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import zarr
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

# Keep concurrent chunk reads bounded.
zarr.config.set({"async.concurrency": 1})

forecast_15d_01 = dynamical_catalog.open(
    "google-weathernext3-forecast-15-day-0-1-degree-virtual", chunks=None
)
forecast_15d_005 = dynamical_catalog.open(
    "google-weathernext3-forecast-15-day-0-05-degree-virtual", chunks=None
)
forecast_48h_01 = dynamical_catalog.open(
    "google-weathernext3-forecast-48-hour-0-1-degree-virtual", chunks=None
)
forecast_48h_005 = dynamical_catalog.open(
    "google-weathernext3-forecast-48-hour-0-05-degree-virtual", chunks=None
)
forecast_15d_01

The dimensions are `init_time`, `statistic`, `lead_time`, `y` and `x`. Native `y` is latitude ascending south to north, and `x` is longitude on 0–360°; 105°W is `x=255`.

Select `statistic="mean"` explicitly. Mean and p10/p25/p50/p75/p90 are precomputed summaries, not individual members to average together. Temperature reads in degrees Celsius, so no Kelvin conversion is needed; recent initializations can contain NaNs while source files arrive.

## What changes when a newer initialization is available?

Compare the October 21 00:00 UTC 15-day forecast at +24 h with the newer 11:00 UTC 48-hour forecast at +13 h, both valid October 22 at 00:00 UTC. Each has a 0.1° and a station-trained 0.05° temperature head; these are model fields, not observations. At a shared initialization and lead the two horizons use the same source fields, so these panels instead illustrate the choice of initialization and head.

In [ ]:
forecasts = {
    "15 day, 0.1°": forecast_15d_01.sel(init_time="2026-10-21T00", lead_time="24h"),
    "15 day, 0.05°": forecast_15d_005.sel(init_time="2026-10-21T00", lead_time="24h"),
    "48 hour, 0.1°": forecast_48h_01.sel(init_time="2026-10-21T11", lead_time="13h"),
    "48 hour, 0.05°": forecast_48h_005.sel(init_time="2026-10-21T11", lead_time="13h"),
}
selection = {
    "statistic": "mean",
    "y": slice(25, 55),
    "x": slice(230, 300),
}
temperatures = {
    name: ds["temperature_2m"].sel(**selection).load() for name, ds in forecasts.items()
}

fig, axes = plt.subplots(
    2, 2, figsize=(12, 7), subplot_kw={"projection": ccrs.PlateCarree()}
)
for ax, (name, temperature) in zip(axes.flat, temperatures.items()):
    temperature.plot.pcolormesh(
        ax=ax,
        x="x",
        y="y",
        transform=ccrs.PlateCarree(),
        cmap="coolwarm",
        vmin=-10,
        vmax=35,
        center=False,
    )
    ax.coastlines()
    init_hour = pd.Timestamp(temperature.init_time.item()).hour
    ax.set_title(f"{name} — init Oct 21 {init_hour:02d} UTC")
fig.suptitle("Mean temperature valid October 22, 2026 at 00:00 UTC")
fig.tight_layout()

## What does the nearest grid cell to Denver represent?

Reuse the fields above to compare the resolved cells near Denver. The native grids need not select the same location, and terrain matters when comparing a forecast grid cell with a station observation.

In [ ]:
denver = {
    name: field.sel(y=39.74, x=(-104.99) % 360, method="nearest")
    for name, field in temperatures.items()
}
pd.DataFrame(
    [
        {
            "product": name,
            "latitude": float(point.y),
            "longitude_0_360": float(point.x),
            "temperature_C": float(point),
        }
        for name, point in denver.items()
    ]
).set_index("product")

## How does the forecast pattern evolve?

Sample the 48-hour 0.1° forecast initialized October 21 at 11:00 UTC at +4, +7, +10 and +13 h, reusing the final field already loaded. The animation shows four forecast frames three hours apart; it does not interpolate the hourly evolution between them.

In [ ]:
earlier_temperature = (
    forecast_48h_01["temperature_2m"]
    .sel(
        **selection,
        init_time="2026-10-21T11",
        lead_time=pd.to_timedelta([4, 7, 10], unit="h"),
    )
    .load()
)
temperature_frames = xr.concat(
    [earlier_temperature, temperatures["48 hour, 0.1°"]],
    dim="lead_time",
    coords="minimal",
    compat="equals",
    join="exact",
)

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})
mesh = temperature_frames.isel(lead_time=0).plot.pcolormesh(
    ax=ax,
    x="x",
    y="y",
    transform=ccrs.PlateCarree(),
    cmap="coolwarm",
    vmin=-10,
    vmax=35,
    center=False,
)
ax.coastlines()


def update(frame: int) -> tuple:
    field = temperature_frames.isel(lead_time=frame)
    mesh.set_array(field.values.ravel())
    valid_time = pd.Timestamp(field.init_time.item()) + pd.Timedelta(
        field.lead_time.item()
    )
    ax.set_title(f"48 hour, 0.1° mean temperature — {valid_time:%Y-%m-%d %H:%M} UTC")
    return (mesh,)


anim = FuncAnimation(fig, update, frames=4, interval=700)
plt.close(fig)
HTML(anim.to_jshtml())

## Community challenge

Consult each catalog validation report for its sampled scope and review notes, including sea-surface temperature, before drawing conclusions about accuracy. Forecast differences and percentile spread are not by themselves skill or calibration scores.

- How do the two temperature heads compare with independent observations at matched times and resolved grid cells?
- How do the precomputed percentile fields differ from the mean at a location of interest?
- When does the longer forecast horizon become more useful than the more frequent initializations?